[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/personal-health-agent/binf4070-lab-2026/blob/main/week-01/lab.ipynb)

# Week 1 Lab: Colab, Health-Data Streams, and a First API Call

**Course:** BINF 4070 — The Future of Personal Health Assistant<br>
**Week:** 1 · Shopping-period onboarding<br>

This lab gets us comfortable with the tools we will use throughout
the course. We will run Python in Google Colab, inspect three small **synthetic**
health-data streams, make plots, and see how a short OpenAI API request is built.

By the end, you will be able to:

- run and rerun Colab text and code cells while keeping track of runtime state;
- inspect, filter, summarize, and plot pandas DataFrames;
- recognize that a data gap is not the same as a measured zero;
- read an offline map of a synthetic GPS trace; and
- distinguish a system prompt from a user prompt in the Responses API.

All people, measurements, locations, and routines in this notebook are
synthetic.


In [ ]:
# Install the major package versions used in this lab, then import them.
%pip install -q "numpy>=2,<3" "pandas>=2,<3" "matplotlib>=3,<4" "openai>=2,<3"

import json
import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

print("Environment ready")
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)


---

## Part 1: Get comfortable with Google Colab

If you are already familiar with Google Colab and Jupyter notebooks, you can
skim Part 1.

A notebook mixes two kinds of cells:

- A **text cell** contains headings, explanations, links, and instructions.
- A **code cell** contains Python. Its output appears directly below the cell.

To run a code cell, click its play button or place the cursor in it and press
**Shift+Enter**. The square brackets at the left show whether a cell is waiting,
running, or finished.

If you opened this notebook from the badge, choose **File → Save a copy in
Drive** before making changes. Your copy is the one you should edit and keep.


### 1.1 Run a code cell and read its output

Run the next cell. Colab sends the Python code to a temporary **runtime** and
shows the result below it. The last expression is displayed automatically;
`print(...)` lets us label output explicitly.


In [ ]:
course_name = "BINF 4070"
week_number = 1

print("Course:", course_name)
print("Week:", week_number)
2 + 3


### 1.2 Variables keep values in runtime memory

Python variables can hold text, numbers, lists, dictionaries, DataFrames, and
many other objects. Later cells can use a variable after the cell that creates
it has run.


In [ ]:
synthetic_snapshot = {
    "day": "Friday",
    "steps": [420, 860, 1250],
    "note": "Synthetic classroom example",
}

three_hour_total = sum(synthetic_snapshot["steps"])
print(synthetic_snapshot["note"])
print("Steps across three example hours:", three_hour_total)


### 1.3 Cell order and runtime state matter

The runtime remembers what has already run, not what merely appears above on
the page. Run the next cell more than once and watch the counter change. This is
useful for experimentation, but it also explains why running cells out of order
can produce confusing results.


In [ ]:
run_count = globals().get("run_count", 0) + 1
print("This cell has run", run_count, "time(s) in the current runtime.")


### 1.4 Restart, reconnect, and rerun

A Colab runtime is temporary. **Runtime → Restart session** clears variables,
imports, and generated data. After a restart or disconnect, begin at the setup
cell and choose **Runtime → Run all**, or run the cells from top to bottom.

We will keep going now so the variables remain available. At the end of class,
you can practice restarting and rerunning your saved copy. If a later cell says
`NameError`, the usual cause is that the cell creating that variable has not run
in the current runtime.


In [ ]:
# A fully worked check: the variables created above are available.
assert course_name == "BINF 4070"
assert three_hour_total == 2530
assert run_count >= 1
print("Part 1 runtime check passed.")


---

## Part 2: Explore synthetic health-data streams with pandas and matplotlib

Wearables and phones produce **longitudinal streams**: repeated observations
indexed by time. We will generate three small examples inside the notebook:

1. hourly step estimates;
2. 5-minute heart-rate estimates with one missing interval; and
3. a synthetic GPS trace around Columbia University's Morningside campus.

The examples are deliberately inspectable: we can see each timestamp, value,
transformation, and gap. A displayed step or heart-rate value is an estimate
produced downstream of a sensor. The GPS labels describe a fictional itinerary.
None of these data supports a medical conclusion about a real person.


In [ ]:
# Generate all three datasets deterministically: every clean run gives the same data.
RNG = np.random.default_rng(4070)
TIMEZONE = "America/New_York"
LAB_DAY = "2026-09-11"

# 1) Hourly step estimates for one synthetic day.
hourly_index = pd.date_range(f"{LAB_DAY} 00:00", periods=24, freq="h", tz=TIMEZONE)
base_steps = np.array([
    0, 0, 0, 0, 0, 15, 90, 610, 980, 340, 520, 860,
    1110, 560, 280, 730, 890, 450, 940, 670, 310, 130, 35, 0,
])
step_jitter = RNG.integers(-35, 36, size=24)
hourly_steps = np.maximum(base_steps + step_jitter, 0).astype(int)

def activity_period(hour):
    if hour < 6:
        return "overnight"
    if hour < 12:
        return "morning"
    if hour < 17:
        return "afternoon"
    if hour < 22:
        return "evening"
    return "late evening"

steps_df = pd.DataFrame({
    "timestamp": hourly_index,
    "steps": hourly_steps,
})
steps_df["activity_period"] = [activity_period(ts.hour) for ts in steps_df["timestamp"]]

# 2) Five-minute heart-rate estimates. Six expected rows are intentionally absent.
hr_index = pd.date_range(
    f"{LAB_DAY} 07:00", f"{LAB_DAY} 22:55", freq="5min", tz=TIMEZONE
)
decimal_hour = hr_index.hour + hr_index.minute / 60
daily_wave = 4 * np.sin((decimal_hour - 8) / 14 * np.pi)
movement_boost = np.select(
    [
        (decimal_hour >= 8.0) & (decimal_hour < 9.0),
        (decimal_hour >= 12.0) & (decimal_hour < 13.0),
        (decimal_hour >= 17.0) & (decimal_hour < 18.0),
    ],
    [18, 12, 20],
    default=0,
)
hr_values = np.rint(66 + daily_wave + movement_boost + RNG.normal(0, 2.4, len(hr_index))).astype(int)
heart_rate_full = pd.DataFrame({"timestamp": hr_index, "heart_rate_bpm": hr_values})
gap_start = pd.Timestamp(f"{LAB_DAY} 13:30", tz=TIMEZONE)
gap_end = pd.Timestamp(f"{LAB_DAY} 13:55", tz=TIMEZONE)
gap_mask = heart_rate_full["timestamp"].between(gap_start, gap_end)
heart_rate_df = heart_rate_full.loc[~gap_mask].reset_index(drop=True)

# 3) A one-day GPS trace through a fictional student itinerary.
waypoints = pd.DataFrame(
    [
        ("07:30", "Residence hall", 40.80670, -73.96265),
        ("08:20", "Classroom", 40.80915, -73.96090),
        ("10:15", "Coffee shop", 40.80540, -73.96500),
        ("11:25", "Restaurant", 40.80495, -73.96425),
        ("12:50", "Classroom", 40.80915, -73.96090),
        ("16:20", "College Walk", 40.80765, -73.96205),
        ("18:00", "Restaurant", 40.80565, -73.96515),
        ("20:10", "Residence hall", 40.80670, -73.96265),
    ],
    columns=["clock_time", "place", "latitude", "longitude"],
)
waypoints["timestamp"] = pd.to_datetime(
    LAB_DAY + " " + waypoints["clock_time"]
).dt.tz_localize(TIMEZONE)

gps_index = pd.date_range(
    waypoints["timestamp"].min(), waypoints["timestamp"].max(), freq="5min"
)
route_points = waypoints.set_index("timestamp")[["latitude", "longitude"]]
combined_index = gps_index.union(route_points.index)
interpolated_route = (
    route_points.reindex(combined_index)
    .sort_index()
    .interpolate(method="time")
    .reindex(gps_index)
)
gps_df = interpolated_route.reset_index(names="timestamp")
gps_df["latitude"] += RNG.normal(0, 0.000025, len(gps_df))
gps_df["longitude"] += RNG.normal(0, 0.000025, len(gps_df))

print("Synthetic datasets created:")
print("  hourly steps:", steps_df.shape)
print("  5-minute heart rate:", heart_rate_df.shape)
print("  GPS samples:", gps_df.shape)


In [ ]:
# DataFrames have rows, columns, data types, and an index.
display(steps_df.head(4))
display(heart_rate_df.head(4))
display(gps_df.head(4))

print("Step columns:", steps_df.columns.tolist())
print("Step data types:")
print(steps_df.dtypes)


### 2.1 Hourly steps: filter, group, summarize, and plot

The next worked example uses common pandas operations:

- `loc[...]` filters rows;
- `groupby(...)` combines rows that share a category;
- `agg(...)` computes summaries; and
- `plot(...)` sends columns to matplotlib.

The values are hourly estimates, not raw accelerometer signals.


In [ ]:
# Filter to hours with at least 500 estimated steps.
active_hours = steps_df.loc[
    steps_df["steps"] >= 500,
    ["timestamp", "steps", "activity_period"],
]
display(active_hours)

# Group by a simple time-of-day label.
period_summary = (
    steps_df.groupby("activity_period", sort=False)
    .agg(total_steps=("steps", "sum"), mean_hourly_steps=("steps", "mean"))
    .round(1)
)
display(period_summary)

# Plot the full day.
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(steps_df["timestamp"].dt.hour, steps_df["steps"], color="#2E75B6")
ax.set(title="Synthetic hourly step estimates", xlabel="Hour of day", ylabel="Estimated steps")
ax.set_xticks(range(0, 24, 2))
ax.grid(axis="y", alpha=0.25)
plt.show()


#### Try it: find the busiest hours

Use `nlargest(3, "steps")` to select the three rows with the largest step
counts. Keep `timestamp`, `steps`, and `activity_period`. Also calculate the
full-day total with `.sum()`.

Expected result: a three-row DataFrame ordered from highest to lower steps, plus
one integer daily total.


In [ ]:
# TODO: Find the three busiest hours and the full-day step total.
# Hint: steps_df.nlargest(3, "steps") selects the three largest rows.
top_step_hours = None
daily_step_total = None

raise NotImplementedError("Complete the busiest-hours exercise, then rerun this cell.")


In [ ]:
# Verify the busiest-hours exercise.
assert isinstance(top_step_hours, pd.DataFrame), "top_step_hours must be a DataFrame."
assert top_step_hours.shape == (3, 3), "Keep exactly 3 rows and 3 requested columns."
assert top_step_hours["steps"].is_monotonic_decreasing, "Order rows from highest to lower steps."
assert daily_step_total == int(steps_df["steps"].sum()), "Recheck the full-day sum."
print("Busiest-hours checks passed.")


In [ ]:
# TODO: In 2–3 sentences, cite one high-activity hour and one low-activity hour from your table or plot. Describe the pattern without making a medical interpretation.
step_pattern_note = "TODO: replace this placeholder with your answer."
print(step_pattern_note)


### 2.2 Five-minute heart rate: reveal missingness before interpreting a trend

The table contains heart-rate **estimates** every five minutes from 7:00 a.m.
through 10:55 p.m., except for one realistic missing interval. An absent record
could reflect charging, removal, failed contact, or synchronization. It does not
mean the heart rate was zero.

First, we summarize the rows that are present. Then we construct the full
expected five-minute timeline so the gap becomes visible.


In [ ]:
# Summarize available estimates by hour.
hourly_hr = (
    heart_rate_df.set_index("timestamp")
    .resample("1h")
    .agg(mean_bpm=("heart_rate_bpm", "mean"), observed_samples=("heart_rate_bpm", "count"))
    .round({"mean_bpm": 1})
)
display(hourly_hr)

print("Available five-minute rows:", len(heart_rate_df))
print("Observed range:", heart_rate_df["heart_rate_bpm"].min(), "to", heart_rate_df["heart_rate_bpm"].max(), "bpm")


#### Try it: expose the missing rows

Create an expected five-minute index from the first through last timestamp. Set
the observed timestamps as the DataFrame index, then use `.reindex(...)` to add
the absent rows as `NaN` values. Count the missing estimates.

Expected result: `missing_interval_count` is `6`, representing a 30-minute gap.


In [ ]:
# TODO: Reindex the observed rows onto the expected five-minute timeline.
# Hint 1: pd.date_range(first_timestamp, last_timestamp, freq="5min")
# Hint 2: heart_rate_df.set_index("timestamp").reindex(expected_hr_index)
expected_hr_index = None
heart_rate_regular = None
missing_times = None
missing_interval_count = None

raise NotImplementedError("Complete the missing-interval exercise, then rerun this cell.")


In [ ]:
# Verify that the complete timeline reveals the designed gap.
assert isinstance(heart_rate_regular, pd.DataFrame), "heart_rate_regular must be a DataFrame."
assert len(heart_rate_regular) == len(expected_hr_index), "The table must use the expected index."
assert missing_interval_count == 6, "Expected six missing five-minute estimates."
assert missing_times[0].strftime("%H:%M") == "13:30"
assert missing_times[-1].strftime("%H:%M") == "13:55"
print("Missingness checks passed.")


In [ ]:
# TODO: In 2–3 sentences, explain why the six missing rows should not be filled with zero. Name at least one plausible data-collection reason for the gap.
heart_rate_gap_explanation = "TODO: replace this placeholder with your answer."
print(heart_rate_gap_explanation)


In [ ]:
# Plot the reindexed series. Matplotlib leaves a visible break where values are NaN.
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(
    heart_rate_regular.index,
    heart_rate_regular["heart_rate_bpm"],
    color="#E67E22",
    linewidth=1.5,
)
ax.axvspan(missing_times.min(), missing_times.max(), color="#BDBDBD", alpha=0.35, label="No observation")
ax.set(title="Synthetic 5-minute heart-rate estimates", xlabel="Time", ylabel="Estimated bpm")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


### 2.3 GPS: place coordinates on a simple offline map

Latitude and longitude observations can support behavioral features such as
time away from a usual location or travel between areas. Those features still
do not reveal *why* a person moved. Here we already know the fictional itinerary
because we generated it.

The background below is a schematic teaching map, not a navigation map. It uses
only matplotlib shapes and labels, so it needs no map tile, location service, or
network request.


In [ ]:
# Plot a schematic campus context, the synthetic trace, and labeled itinerary stops.
fig, ax = plt.subplots(figsize=(8.5, 7.5))

# Schematic campus block and nearby streets.
campus_x = [-73.96365, -73.95910, -73.95910, -73.96365, -73.96365]
campus_y = [40.80665, 40.80665, 40.81085, 40.81085, 40.80665]
ax.fill(campus_x, campus_y, color="#DCEAF7", alpha=0.85, label="Columbia campus area (schematic)")
ax.axvline(-73.96375, color="#B0B0B0", linewidth=3)
ax.axvline(-73.95900, color="#B0B0B0", linewidth=3)
ax.axhline(40.80500, color="#D0D0D0", linewidth=2)
ax.axhline(40.80865, color="#D0D0D0", linewidth=2)
ax.axhline(40.81125, color="#D0D0D0", linewidth=2)
ax.text(-73.96382, 40.81155, "Broadway", rotation=90, va="top", ha="right", color="#666666")
ax.text(-73.95893, 40.81155, "Amsterdam Ave", rotation=90, va="top", ha="left", color="#666666")
ax.text(-73.96595, 40.80873, "W 116th St", color="#777777")

# GPS trace and unique labeled stops.
ax.plot(gps_df["longitude"], gps_df["latitude"], color="#2E75B6", linewidth=2, label="Synthetic GPS trace")
unique_stops = waypoints.drop_duplicates(subset=["place"], keep="first")
ax.scatter(unique_stops["longitude"], unique_stops["latitude"], s=60, color="#E67E22", zorder=3)

label_offsets = {
    "Residence hall": (5, -15),
    "Classroom": (6, 7),
    "Coffee shop": (-72, 8),
    "Restaurant": (-72, -16),
    "College Walk": (7, -18),
}
for row in unique_stops.itertuples():
    ax.annotate(
        row.place,
        (row.longitude, row.latitude),
        xytext=label_offsets[row.place],
        textcoords="offset points",
        fontsize=9,
        arrowprops={"arrowstyle": "-", "color": "#777777", "lw": 0.8},
    )

ax.set(
    title="Synthetic one-day GPS trace near Columbia University",
    xlabel="Longitude",
    ylabel="Latitude",
    xlim=(-73.9662, -73.9578),
    ylim=(40.8042, 40.8120),
)
ax.set_aspect(1 / np.cos(np.deg2rad(40.81)))
ax.ticklabel_format(style="plain", useOffset=False)
ax.grid(alpha=0.15)
ax.legend(loc="upper left")
plt.show()

display(waypoints[["timestamp", "place", "latitude", "longitude"]])


In [ ]:
# TODO: Choose one transition in the waypoint table and cite its two places and times. Write one thing the location data reveal. Then write one thing they cannot reveal about the person's actions or experience.
gps_observation_response = "TODO: replace this placeholder with your answer."
print(gps_observation_response)


---

## Part 3: Make a first OpenAI Responses API call

These steps assume that `OPENAI_API_KEY` has been added to your Colab
environment. If you have your own API key, you may use it. Beginning in Week 03,
you will receive a new API key for this class.

In Colab, a secret is stored outside the notebook code. Open the **Secrets**
panel with the key icon, create a secret named `OPENAI_API_KEY`, and enable
notebook access. Never paste a key into a code cell, print it, or include it in a
submitted notebook.

This section runs cleanly without a key: each live demonstration prints a skip
message instead. API output can vary, and a successful request does not prove
that a response is accurate.

We will use the current Python SDK's **Responses API** and the cost-conscious
`gpt-5.6-luna` model. Each request sets `store=False`, uses no reasoning effort,
and caps the output length.

Official references: [Python quickstart](https://developers.openai.com/api/docs/quickstart),
[text generation with the Responses API](https://developers.openai.com/api/docs/guides/text),
and [`gpt-5.6-luna`](https://developers.openai.com/api/docs/models/gpt-5.6-luna).


In [ ]:
from openai import OpenAI

def read_openai_api_key():
    # Read a Colab Secret first, then use a local environment-variable fallback.
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key
    except Exception:
        # Outside Colab, or when the secret is absent/not shared with this notebook.
        pass
    return os.environ.get("OPENAI_API_KEY")


OPENAI_API_KEY_VALUE = read_openai_api_key()
openai_client = (
    OpenAI(api_key=OPENAI_API_KEY_VALUE, timeout=30.0, max_retries=0)
    if OPENAI_API_KEY_VALUE
    else None
)
HAVE_OPENAI_KEY = openai_client is not None

print("OpenAI key available:", HAVE_OPENAI_KEY)
if not HAVE_OPENAI_KEY:
    print("Live calls will be skipped; the rest of the notebook remains complete.")


def request_demo_text(system_prompt, user_prompt, max_output_tokens=80):
    # Make one bounded call, or skip safely when no key is available.
    if openai_client is None:
        print("OpenAI call skipped because OPENAI_API_KEY is not available.")
        return None

    try:
        response = openai_client.responses.create(
            model="gpt-5.6-luna",
            instructions=system_prompt,
            input=user_prompt,
            reasoning={"effort": "none"},
            max_output_tokens=max_output_tokens,
            store=False,
        )
        print(response.output_text)
        return response.output_text
    except Exception as exc:
        print(f"OpenAI call did not complete ({type(exc).__name__}). Check the key and try again.")
        return None


### 3.1 Two tiny requests

For the Responses API:

- `instructions=` carries the **system prompt**: the role, boundaries, and
  response rules for this request.
- `input=` carries the **user prompt**: the immediate question, task, or data.

The first request sends `Hello, world`. The second asks `Tell me who you are`.
The system prompt asks for a short classroom-demo response in both cases.


In [ ]:
hello_system_prompt = "You are a concise classroom demo assistant. Reply in one sentence."
hello_user_prompt = "Hello, world"

hello_output = request_demo_text(
    system_prompt=hello_system_prompt,
    user_prompt=hello_user_prompt,
    max_output_tokens=60,
)


In [ ]:
identity_system_prompt = (
    "You are a concise classroom demo assistant. "
    "Describe your role and one limitation in at most two sentences."
)
identity_user_prompt = "Tell me who you are"

identity_output = request_demo_text(
    system_prompt=identity_system_prompt,
    user_prompt=identity_user_prompt,
    max_output_tokens=80,
)


### 3.2 Ask for a plain-language summary of the synthetic step pattern

An API call should receive the smallest amount of data needed for its task.
Rather than sending every row, we compute an inspectable daily total, the peak
hour, and time-of-day totals with pandas. The model is asked to restate those
facts, avoid medical claims, and name a limitation.


In [ ]:
# Build a compact, inspectable summary from Part 2 without sending any personal data.
peak_row = steps_df.loc[steps_df["steps"].idxmax()]
period_totals = (
    steps_df.groupby("activity_period", sort=False)["steps"]
    .sum()
    .astype(int)
    .to_dict()
)

step_summary_record = {
    "data_status": "synthetic classroom example",
    "date": LAB_DAY,
    "daily_total_steps": int(steps_df["steps"].sum()),
    "peak_hour": peak_row["timestamp"].strftime("%H:%M"),
    "peak_hour_steps": int(peak_row["steps"]),
    "period_totals": period_totals,
}
print(json.dumps(step_summary_record, indent=2))

summary_system_prompt = (
    "You summarize synthetic activity data for a classroom demonstration. "
    "Use only the supplied values. Write three short bullets, state that the data are synthetic, "
    "make no medical or fitness judgment, and mention that hourly totals hide within-hour detail."
)
summary_user_prompt = "Summarize this daily step pattern:\n" + json.dumps(step_summary_record)

step_summary_output = request_demo_text(
    system_prompt=summary_system_prompt,
    user_prompt=summary_user_prompt,
    max_output_tokens=140,
)


---

## Take-home TODO: Explore one synthetic data stream

Choose whichever Part 2 dataset interests you most: `steps_df`,
`heart_rate_regular`, or `gps_df`. Use it to answer one small question of your
own with a new plot.

- State the dataset and question you chose.
- Make one plot with a specific title and labeled axes.
- Write one thing the plot reveals and one thing it cannot reveal.

This take-home does not require an API call.


In [ ]:
# TODO: Choose one Part 2 dataset and answer one small question with a new plot.
takehome_dataset_name = "TODO: steps_df, heart_rate_regular, or gps_df"
takehome_question = "TODO: Write one question that your plot can help answer."

# TODO: Create your plot below. Include a specific title and labeled axes.

takehome_what_plot_reveals = "TODO: Write one thing your plot reveals."
takehome_what_plot_cannot_reveal = "TODO: Write one thing your plot cannot reveal."

print("Dataset:", takehome_dataset_name)
print("Question:", takehome_question)
print("What the plot reveals:", takehome_what_plot_reveals)
print("What the plot cannot reveal:", takehome_what_plot_cannot_reveal)

raise NotImplementedError("Complete the take-home plot and two short answers, then rerun this cell.")


---

## Troubleshooting

| What you see | What to try |
|---|---|
| `NameError` after a reconnect or restart | Run the setup and data-generation cells again, then continue from top to bottom. |
| A package import fails | Rerun the setup cell and wait for installation to finish. |
| A TODO cell raises `NotImplementedError` | Complete the named variables, remove the placeholder line, and rerun the cell plus its check. |
| A plot is missing | Run the cell that creates its DataFrame first, then rerun the plotting cell. |
| `OpenAI key available: False` | Add `OPENAI_API_KEY` in Colab Secrets and enable notebook access, or continue with the clean skip path. |
| An OpenAI request does not complete | Check secret access, quota, and the network. The API demonstrations are not required for the data exercises. |

Do not paste an API key into the notebook while troubleshooting. The notebook
never needs device data or identifiable health data.


## TODO: Summary and reflection

Did you use AI for this lab? If so, how?


In [ ]:
# TODO: Did you use AI for this lab? If so, how?
week01_ai_reflection = "TODO: Replace this placeholder with your answer."

print("AI-use reflection:", week01_ai_reflection)
